Preprocessing for TRT (gctg-clean)

This pipeline measures total reading time (TRT) per word AOI per subject using the gctg-clean dataset


In [1]:
# Install dependencies for reproducibility
%pip install -q pymovements polars matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.


In [2]:
from pathlib import Path
import re
import polars as pl
import pymovements as pm

BASE = Path("data-clean")
PROCESSED_DIR = BASE / "processed"
CACHE_DIR = PROCESSED_DIR / "cache"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Helpers

def is_practice(name: str) -> bool:
    return "practice" in name.lower()


def extract_condition(name: str) -> str:
    lname = name.lower()
    if "neg" in lname:
        return "neg"
    if "pos" in lname:
        return "pos"
    if "zero" in lname:
        return "zero"
    return "unknown"

In [3]:
# Load clean dataset (as per lecturer's notebook, but using gctg-clean)
dataset = pm.Dataset("gctg-clean.yaml", str(BASE)).download().load()
# Keep dataset as-is (no split); we'll handle stimulus grouping later
print(f"Loaded gaze frames: {len(dataset.gaze)}")

INFO:pymovements.dataset.dataset:        You are downloading the gctg dataset. Please be aware that pymovements does not
        host or distribute any dataset resources and only provides a convenient interface to
        download the public dataset resources that were published by their respective authors.

        Please cite the referenced publication if you intend to use the dataset in your research.
        


Using already downloaded and verified file: data-clean\downloads\gctg-data-clean.zip
Extracting gctg-data-clean.zip to data-clean\raw


100%|██████████| 822/822 [00:02<00:00, 300.83it/s]


  0%|          | 0/12 [00:00<?, ?it/s]

Loaded gaze frames: 12


In [4]:
# Collect stimuli and exclude practice
all_samples = pl.concat([g.samples for g in dataset.gaze])
stimulus_names = all_samples["stimulus"].unique().to_list()
stimulus_names = [s for s in stimulus_names if not is_practice(s)]
subjects = all_samples["subject_id"].unique().to_list()
print(f"Stimuli (non-practice): {len(stimulus_names)} | Subjects: {len(subjects)} -> {sorted(subjects)}")
stimulus_names[:10]

Stimuli (non-practice): 152 | Subjects: 12 -> ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12']


['prize-pos.interest',
 'voicemail-zero.question',
 'delayed-pos.question',
 'goldfish-zero.difficulty',
 'voicemail-neg.text.3',
 'blackout-zero.text.3',
 'goldfish-zero.question',
 'breakfast-pos.question',
 'blackout-neg.text.3',
 'voicemail-zero.naturalness']

In [5]:
# Load word AOIs for each stimulus from the clean package
stimuli = {}
missing_aois = []
for stimulus_name in stimulus_names:
    aois_path = BASE / "raw" / "stimuli" / f"{stimulus_name}.word.csv"
    if not aois_path.exists():
        missing_aois.append(stimulus_name)
        continue
    aois = pl.read_csv(aois_path)
    stimulus = pm.stimulus.TextStimulus(
        aois,
        aoi_column="content",
        start_x_column="left",
        start_y_column="top",
        end_x_column="right",
        end_y_column="bottom",
    )
    stimuli[stimulus_name] = stimulus
print(f"Loaded AOIs for {len(stimuli)} stimuli. Missing AOIs: {len(missing_aois)}")

Loaded AOIs for 152 stimuli. Missing AOIs: 0


In [6]:
# Event detection using IDT defaults, then fixation locations in pixel space
# (matches lecturer’s pipeline: pix2deg -> detect("idt") -> compute_properties("location", pixel))
dataset.pix2deg()
dataset.detect("idt", clear=True)
dataset.compute_properties(("location", {"position_column": "pixel"}))
print("Events detected and fixation locations computed.")

  0%|          | 0/12 [00:00<?, ?it/s]

0it [00:00, ?it/s]

C:\Python312\Lib\site-packages\pymovements\events\detection\_idt.py:48: RuntimeWarning: All-NaN slice encountered
  return sum(np.nanmax(positions, axis=0) - np.nanmin(positions, axis=0))


  0%|          | 0/12 [00:00<?, ?it/s]

Events detected and fixation locations computed.


In [7]:
# Deprecated mapping approach (kept as no-op to maintain cell order)
# Mapping to AOIs is handled later via explicit per-subject-per-stimulus processing.
len([])

0

In [8]:
# Quick head (does not modify outputs)
trt.sample(n=min(5, trt.height)) if 'trt' in locals() else None

In [9]:
# Map fixations to word AOIs via point-in-rectangle and compute TRT reliably

def map_events_to_aois(events_df: pl.DataFrame, aoi_df: pl.DataFrame) -> pl.DataFrame:
    # events_df has 'location' as [x, y] list; expand to columns
    ev = events_df.with_columns([
        pl.col("location").list.first().alias("x"),
        pl.col("location").list.last().alias("y"),
    ])
    # Ensure AOI columns exist
    aois = aoi_df.select([
        pl.col("index"),
        pl.col("content"),
        pl.col("left"),
        pl.col("right"),
        pl.col("top"),
        pl.col("bottom"),
    ])
    # Cross-join and filter by within-rect; for performance we can prefilter by x/y bounds
    ev_small = ev.select(["subject_id", "stimulus", "duration", "x", "y"])  # keep necessary cols
    joined = ev_small.join(aois, how="cross")
    mapped = joined.filter(
        (pl.col("x") >= pl.col("left")) & (pl.col("x") <= pl.col("right")) &
        (pl.col("y") >= pl.col("top")) & (pl.col("y") <= pl.col("bottom"))
    )
    # Aggregate TRT by AOI
    trt = (
        mapped
        .group_by(["subject_id", "stimulus", "index", "content"]) 
        .agg(pl.col("duration").sum().alias("total_reading_time"))
    )
    # Right join to include AOIs with TRT=0
    # Use stimulus-wide AOIs; fill subject_id/stimulus later
    trt_full = (
        trt.join(aois.select(["index", "content"]), on=["index", "content"], how="right")
        .with_columns(pl.col("total_reading_time").fill_null(0))
    )
    # Fill subject and stimulus ids (unique within events)
    subj = events_df["subject_id"].unique().item()
    stim = events_df["stimulus"].unique().item()
    trt_full = trt_full.with_columns([
        pl.lit(subj).alias("subject_id").cast(pl.Utf8),
        pl.lit(stim).alias("stimulus").cast(pl.Utf8),
        pl.lit(extract_condition(stim)).alias("condition")
    ])
    return trt_full.select(["subject_id", "stimulus", "condition", "index", "content", "total_reading_time"])

In [10]:
# Corrected processing: iterate per subject and per stimulus within subject
trt_tables = []
cache_written = 0
for g in dataset.gaze:
    ev_all = g.events.frame
    subj = ev_all["subject_id"].unique().item()
    stims = ev_all["stimulus"].unique().to_list()
    for stim in stims:
        if is_practice(stim):
            continue
        if stim not in stimuli:
            continue
        sub_ev = ev_all.filter(pl.col("stimulus") == stim)
        # cache raw events with locations
        out_path = CACHE_DIR / f"events_{subj}_{stim}.parquet"
        sub_ev.write_parquet(out_path)
        cache_written += 1
        # compute TRT
        trt_tables.append(map_events_to_aois(sub_ev, stimuli[stim].aois))

print(f"Cached event tables: {cache_written}")
if trt_tables:
    trt = pl.concat(trt_tables, how="vertical_relaxed")
    out_csv = PROCESSED_DIR / "trt_by_word.csv"
    out_parquet = PROCESSED_DIR / "trt_by_word.parquet"
    trt.write_csv(out_csv)
    trt.write_parquet(out_parquet)
    print(f"Saved TRT rows: {trt.height} -> {out_csv}")
else:
    raise RuntimeError("No TRT tables produced. Verify events and AOIs.")

Cached event tables: 555
Saved TRT rows: 36160 -> data-clean\processed\trt_by_word.csv


In [13]:
# Preview a small sample of the final TRT table (non-destructive)
trt.head(10).to_pandas()

,subject_id,stimulus,condition,index,content,total_reading_time
0,P01,blackout-zero.text.2,zero,202,One,187
1,P01,blackout-zero.text.2,zero,203,brave,255
2,P01,blackout-zero.text.2,zero,204,"soul,",0
3,P01,blackout-zero.text.2,zero,205,a,0
4,P01,blackout-zero.text.2,zero,206,young,255
5,P01,blackout-zero.text.2,zero,207,woman,173
6,P01,blackout-zero.text.2,zero,208,named,102
7,P01,blackout-zero.text.2,zero,209,"Sarah,",135
8,P01,blackout-zero.text.2,zero,210,decided,165
9,P01,blackout-zero.text.2,zero,211,to,0


# Linguistic Features Pipeline

This pipeline extracts the following linguistic features:
- **Word frequency**: SUBTLEX-US Zipf scores (lemma-first, then surface)
- **Word length**: Alphabetic character count
- **Dependency distance**: Linear distance to syntactic head (Gibson 2000 locality theory)
- **Syntactic tree depth**: Distance from token to root
- **Integration cost**: Locality-based processing difficulty metric

Features are aligned to AOI tokens and merged with TRT data for mixed-effects modeling.

In [12]:
# Install dependencies with version pins for reproducibility
%pip install -q "spacy==3.7.4" "polars==1.6.0" "pyarrow==15.0.2" "wordfreq==3.1.1"

# Download spaCy model if not present
import subprocess
import sys
try:
    import en_core_web_sm
except ImportError:
    print("Downloading spaCy English model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pymovements 0.23.0 requires polars<1.32,>=1.31.0, but you have polars 1.6.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


C:\Python312\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [14]:
from pathlib import Path
import json
import re
import polars as pl
import spacy

# Paths
BASE = Path("data-clean")
RAW = BASE / "raw"
TEXTS_DIR = RAW / "texts"
STIMULI_DIR = RAW / "stimuli"
PROC = BASE / "processed"
RESOURCES = BASE / "resources"

# Ensure output directory exists
PROC.mkdir(parents=True, exist_ok=True)

# File paths
TRT_PARQUET = PROC / "trt_by_word.parquet"
TRT_CSV = PROC / "trt_by_word.csv"
SUBTLEX_PATH = RESOURCES / "SUBTLEX-US.csv"

print(f"Base directory: {BASE.absolute()}")
print(f"SUBTLEX exists: {SUBTLEX_PATH.exists()}")
print(f"TRT data exists: {TRT_PARQUET.exists() or TRT_CSV.exists()}")

Base directory: c:\a\project\data-clean
SUBTLEX exists: True
TRT data exists: True


In [15]:
# Helper functions
def is_practice(name: str) -> bool:
    return "practice" in name.lower()

def extract_condition(name: str) -> str:
    lname = name.lower()
    if "neg" in lname: return "neg"
    if "pos" in lname: return "pos"  
    if "zero" in lname: return "zero"
    return "unknown"

# Text normalization for alignment: keep apostrophes & hyphens
_punct_re = re.compile(r"[^\w'\-]+", flags=re.UNICODE)

def normalize_token(s: str) -> str:
    """Normalize token for alignment: lowercase; keep apostrophes & hyphens; strip other punct."""
    return _punct_re.sub("", s.lower())

def align_aoi_to_spacy_windowed(aoi_tokens: list[str], doc_tokens: list[str], max_window: int = 2) -> list[int | None]:
    """
    Greedy left-to-right alignment by normalized surface forms.
    Supports concatenating up to `max_window` spaCy tokens to match hyphenated/multiword AOIs.
    Also aligns punctuation-only AOIs to identical doc tokens.
    """
    mapping: list[int | None] = [None] * len(aoi_tokens)
    j = 0
    N = len(doc_tokens)

    for i, aoi_tok in enumerate(aoi_tokens):
        raw = aoi_tok.strip()
        tgt = normalize_token(aoi_tok)

        # Handle pure punctuation AOIs by literal match
        if tgt == "" and raw:
            while j < N and doc_tokens[j].strip() != raw:
                j += 1
            if j < N and doc_tokens[j].strip() == raw:
                mapping[i] = j
                j += 1
            continue

        if tgt == "":
            # empty after normalization; skip
            continue

        matched = False
        k = j
        while k < N and not matched:
            for w in range(1, max_window + 1):
                if k + w > N:
                    break
                window_norm = "".join(normalize_token(t) for t in doc_tokens[k:k + w])
                if window_norm == tgt:
                    mapping[i] = k
                    j = k + w
                    matched = True
                    break
            if not matched:
                k += 1

        if not matched:
            j = min(j + 1, N)

    return mapping

In [16]:
# Load TRT data to get stimulus list (excludes practice)
if TRT_PARQUET.exists():
    trt = pl.read_parquet(TRT_PARQUET)
else:
    trt = pl.read_csv(TRT_CSV)

# Filter out practice stimuli
trt = trt.filter(~pl.col("stimulus").str.contains("practice"))
stimuli_to_process = trt["stimulus"].unique().to_list()

print(f"Stimuli to process (from TRT): {len(stimuli_to_process)}")
print(f"Subjects: {trt['subject_id'].n_unique()}")
print(f"Conditions: {sorted(trt['condition'].unique().to_list())}")

Stimuli to process (from TRT): 152
Subjects: 12
Conditions: ['neg', 'pos', 'zero']


In [17]:
# Load AOI data for alignment
def load_aois(stimulus: str) -> pl.DataFrame | None:
    """Load AOI word-level data for a stimulus."""
    aoi_path = STIMULI_DIR / f"{stimulus}.word.csv"
    if not aoi_path.exists():
        return None
        
    aoi_df = pl.read_csv(aoi_path)
    required_cols = {"index", "content", "left", "right", "top", "bottom"}
    missing_cols = required_cols - set(aoi_df.columns)
    
    if missing_cols:
        raise ValueError(f"AOI file missing columns {missing_cols} for {stimulus}")
        
    return aoi_df.sort("index")  # Ensure consistent token order

# Load AOIs for all stimuli
stimulus_aois: dict[str, pl.DataFrame] = {}
missing_aois = []

for stimulus in stimuli_to_process:
    if is_practice(stimulus):
        continue
        
    aoi_df = load_aois(stimulus)
    if aoi_df is not None:
        stimulus_aois[stimulus] = aoi_df
    else:
        missing_aois.append(stimulus)

print(f"Loaded AOIs: {len(stimulus_aois)} stimuli")
if missing_aois:
    print(f"Missing AOI files: {missing_aois}")

Loaded AOIs: 152 stimuli


In [18]:
# Text loading with fallbacks
def load_text(stimulus: str, aoi_df: pl.DataFrame) -> str:
    """Load stimulus text from JSON, TXT, or reconstruct from AOIs."""
    # Try JSON first
    json_path = TEXTS_DIR / f"{stimulus}.json"
    if json_path.exists():
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
            # Handle common JSON structures
            if isinstance(data, dict):
                for key in ("text", "content", "body"):
                    if key in data and isinstance(data[key], str) and data[key].strip():
                        return data[key]
                # Fallback: first string value
                for value in data.values():
                    if isinstance(value, str) and value.strip():
                        return value
        except (json.JSONDecodeError, UnicodeDecodeError):
            pass
    
    # Try TXT file
    txt_path = TEXTS_DIR / f"{stimulus}.txt" 
    if txt_path.exists():
        try:
            return txt_path.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            pass
    
    # Fallback: reconstruct from AOI content
    return " ".join(aoi_df["content"].to_list())

# Test text loading for first few stimuli
for i, (stimulus, aoi_df) in enumerate(list(stimulus_aois.items())[:3]):
    text = load_text(stimulus, aoi_df)
    print(f"{stimulus}: {len(text)} chars, first 100: {text[:100]!r}")
    if i >= 2:  # Only show first 3
        break

breakfast-zero.question: 156 chars, first 100: 'What caused the narrator to feel uneasy during breakfast? The coffee tasted bitter The waitress acte'
prize-zero.text.4: 380 chars, first 100: '"I don\'t know if I\'m ready for that," Emily said, feeling a sense of trepidation. The woman smiled. '
breakfast-neg.question: 203 chars, first 100: 'What unexpected event happened to the narrator at the diner? The diner ran out of food. A stranger j'


In [19]:
# Initialize spaCy pipeline
print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm", exclude=["ner"])  # Exclude NER for speed

# Ensure sentence segmentation
if not nlp.has_pipe("senter") and not nlp.has_pipe("parser"):
    nlp.add_pipe("sentencizer")

print(f"spaCy pipeline: {nlp.pipe_names}")

# Test parsing
test_text = "The quick brown fox jumps over the lazy dog."
doc = nlp(test_text)
print(f"Test parse: {len(doc)} tokens, {len(list(doc.sents))} sentences")

Loading spaCy model...


C:\Python312\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer']
Test parse: 10 tokens, 1 sentences


In [20]:
# Load SUBTLEX-US frequency data
from math import log10

def load_subtlex(path: Path) -> pl.DataFrame:
    """Load SUBTLEX data with robust column detection and scaling.
    Preference order:
    1) SUBTLWF (freq per million): Zipf = log10(SUBTLWF) + 3
    2) Any 'zipf' column: use as-is
    3) lg10WF: ONLY use as-is (no +3), because it's log10(raw frequency), not per million
       and adding +3 would inflate scales. This keeps it monotonic and comparable.
    """
    if not path.exists():
        raise FileNotFoundError(f"SUBTLEX file not found at {path}")

    df_raw = pl.read_csv(path, separator="\t", infer_schema_length=50000)
    cols_lc = {c.lower(): c for c in df_raw.columns}

    # word/form column
    word_col = None
    for candidate in ["word", "spelling", "lemma", "wordform"]:
        if candidate in cols_lc:
            word_col = cols_lc[candidate]
            break
    if not word_col:
        word_col = df_raw.columns[0]

    # 1) Prefer SUBTLWF if present (per million)
    subtlwf_col = None
    for k in ["subtlwf", "freqpm", "freq_per_million"]:
        if k in cols_lc:
            subtlwf_col = cols_lc[k]
            break

    if subtlwf_col:
        zipf_expr = (
            pl.when(pl.col(subtlwf_col).cast(pl.Float64) > 0)
            .then(pl.col(subtlwf_col).cast(pl.Float64).log10() + 3.0)
            .otherwise(None)
        )
    else:
        # 2) Any 'zipf' column
        zipf_candidates = [c for c in df_raw.columns if "zipf" in c.lower()]
        if zipf_candidates:
            base_col = zipf_candidates[0]
            zipf_expr = pl.col(base_col).cast(pl.Float64)
        else:
            # 3) lg10WF (log10 raw counts): use as-is to avoid artificial offset
            lg10_col = None
            for k in ["lg10wf", "log10wf", "lg10_wf"]:
                if k in cols_lc:
                    lg10_col = cols_lc[k]
                    break
            if lg10_col:
                zipf_expr = pl.col(lg10_col).cast(pl.Float64)
            else:
                raise ValueError("SUBTLEX file must have SUBTLWF, Zipf, or lg10WF column")

    result = (
        df_raw.select([
            pl.col(word_col).str.to_lowercase().alias("form"),
            zipf_expr.alias("zipf"),
        ])
        .filter(pl.col("form").str.len_chars() > 0)
        .filter(pl.col("zipf").is_not_null())
        .group_by("form").agg(pl.col("zipf").max())
    )
    return result

# Load SUBTLEX
print("Loading SUBTLEX-US...")
subtlex = load_subtlex(SUBTLEX_PATH)
print(f"SUBTLEX entries: {subtlex.height}")
print("Sample entries:")
print(subtlex.head().to_pandas())

Loading SUBTLEX-US...
SUBTLEX entries: 74286
Sample entries:
         form      zipf
0  sanctimony  1.602060
1      lowest  3.644439
2    unforced  1.301030
3     dirties  2.000000
4       leper  3.093422


In [21]:
# Dependency Locality Theory functions (Gibson 2000)

def token_depth(token) -> int:
    """Calculate syntactic tree depth: distance from token to root."""
    depth = 0
    current = token
    while current.head != current:  # Until we reach root
        depth += 1
        current = current.head
        if depth > 50:  # Prevent infinite loops
            break
    return depth

def dependency_distance(token) -> int:
    """Calculate linear dependency distance: |position - head_position|."""
    return abs(token.i - token.head.i)

def integration_cost(token) -> float:
    """
    Calculate integration cost based on Gibson 2000 Dependency Locality Theory.
    
    Integration cost reflects processing difficulty due to:
    1. Linear distance to syntactic head
    2. Number of discourse referents between dependent and head
    
    Simplified metric: linear distance weighted by dependency type.
    """
    if token.head == token:  # Root has no integration cost
        return 0.0
    
    dist = dependency_distance(token)
    
    # Weight by dependency relation importance (simplified)
    # More important relations have higher integration costs
    relation_weights = {
        "nsubj": 1.0,     # Subject
        "dobj": 1.0,      # Direct object  
        "prep": 0.8,      # Prepositional
        "amod": 0.6,      # Adjectival modifier
        "advmod": 0.6,    # Adverbial modifier
        "det": 0.3,       # Determiner
        "aux": 0.3,       # Auxiliary
        "punct": 0.1,     # Punctuation
    }
    
    weight = relation_weights.get(token.dep_, 0.5)  # Default weight
    
    # Integration cost = distance * relation_weight
    # Add small penalty for very long distances
    cost = dist * weight
    if dist > 5:
        cost += (dist - 5) * 0.1  # Additional penalty for long dependencies
        
    return cost

def locality_features(token) -> dict:
    """Extract all locality-related features for a token."""
    return {
        "dep_dist": dependency_distance(token),
        "depth": token_depth(token),
        "integration_cost": integration_cost(token),
        "dep_label": token.dep_,
        "pos_tag": token.pos_,
    }

In [22]:
# Frequency lookup function
from wordfreq import zipf_frequency

def lookup_zipf(surface: str, lemma: str = "") -> float | None:
    """
    Look up Zipf frequency score.
    SUBTLEX-first, then wordfreq fallback. Candidates include lemma/surface,
    strip possessive 's, de-hyphenized form, and hyphen parts.
    """
    candidates: list[str] = []
    if lemma:
        candidates.append(lemma)
    if surface:
        candidates.append(surface)
        low = surface.lower()
        # strip possessive
        if low.endswith(("’s", "'s")) and len(surface) > 2:
            candidates.append(surface[:-2])
        # de-hyphenated and parts
        if "-" in surface:
            candidates.append(surface.replace("-", ""))
            candidates.extend([p for p in surface.split("-") if p])

    # Normalize and deduplicate
    norm = []
    seen = set()
    for c in candidates:
        n = normalize_token(c)
        if n and n not in seen:
            seen.add(n)
            norm.append(n)

    # 1) SUBTLEX lookup
    for n in norm:
        hit = subtlex.filter(pl.col("form") == n).select("zipf")
        if hit.height:
            return float(hit.item())

    # 2) wordfreq fallback
    for n in norm:
        z = zipf_frequency(n, "en")
        if z > 0:
            return float(z)

    return None  # Not found

# Test frequency lookup
test_words = ["the", "cat", "quickly", "xyzabc"]
for word in test_words:
    freq = lookup_zipf(word)
    print(f"{word}: {freq}")

the: 7.469073206544746
cat: 4.821709997298376
quickly: 4.751971574736327
xyzabc: None


In [23]:
# Main feature extraction
print("Extracting linguistic features...")

feature_tables = []
coverage_stats = []

for stimulus_idx, (stimulus, aoi_df) in enumerate(stimulus_aois.items()):
    print(f"Processing {stimulus_idx+1}/{len(stimulus_aois)}: {stimulus}")
    
    # Load and parse text
    text = load_text(stimulus, aoi_df)
    doc = nlp(text)
    
    # Get non-space tokens for alignment
    doc_tokens_all = [t for t in doc if not t.is_space]
    doc_tokens_text = [t.text for t in doc_tokens_all]
    
    # Get AOI tokens in order
    aoi_tokens = aoi_df.sort("index")["content"].to_list()
    aoi_indices = aoi_df.sort("index")["index"].to_list()
    
    # Align AOI tokens to spaCy tokens (windowed)
    alignment = align_aoi_to_spacy_windowed(aoi_tokens, doc_tokens_text, max_window=2)
    
    # Calculate coverage
    aligned_count = sum(1 for x in alignment if x is not None)
    coverage = aligned_count / max(1, len(aoi_tokens))
    coverage_stats.append((stimulus, coverage, len(aoi_tokens), aligned_count))
    
    # Extract features for each AOI token
    rows = []
    for aoi_idx, content, spacy_idx in zip(aoi_indices, aoi_tokens, alignment):
        if spacy_idx is None:
            # Unaligned token - compute basic features only
            rows.append({
                "stimulus": stimulus,
                "index": int(aoi_idx),
                "content": content,
                "word_len": sum(c.isalpha() for c in content),
                "freq_zipf": None,
                "dep_dist": None,
                "depth": None,
                "integration_cost": None,
                "dep_label": None,
                "pos_tag": None,
                "lemma": None,
                "sentence_id": None,
                "token_id_sent": None,
            })
            continue
        
        # Get spaCy token
        token = doc_tokens_all[spacy_idx]
        
        # Calculate features
        word_len = sum(c.isalpha() for c in content)
        lemma = token.lemma_ if hasattr(token, 'lemma_') else ""
        freq_zipf = lookup_zipf(content, lemma)
        
        # Locality features  
        locality = locality_features(token)
        
        # Sentence information
        sentence_id = None
        token_id_sent = None
        for sent_idx, sent in enumerate(doc.sents):
            if token.i >= sent.start and token.i < sent.end:
                sentence_id = sent_idx
                token_id_sent = token.i - sent.start
                break
        
        rows.append({
            "stimulus": stimulus,
            "index": int(aoi_idx),
            "content": content,
            "word_len": word_len,
            "freq_zipf": freq_zipf,
            "dep_dist": locality["dep_dist"],
            "depth": locality["depth"], 
            "integration_cost": locality["integration_cost"],
            "dep_label": locality["dep_label"],
            "pos_tag": locality["pos_tag"],
            "lemma": lemma,
            "sentence_id": sentence_id,
            "token_id_sent": token_id_sent,
        })
    
    # Convert to DataFrame
    stimulus_features = pl.from_records(rows)
    feature_tables.append(stimulus_features)

print("Feature extraction complete.")

Extracting linguistic features...
Processing 1/152: breakfast-zero.question
Processing 2/152: prize-zero.text.4
Processing 3/152: breakfast-neg.question
Processing 4/152: delayed-neg.text.1
Processing 5/152: blackout-pos.text.2
Processing 6/152: blackout-zero.text.3
Processing 7/152: delayed-pos.interest
Processing 8/152: voicemail-neg.naturalness
Processing 9/152: voicemail-zero.difficulty
Processing 10/152: goldfish-neg.text.1
Processing 11/152: voicemail-pos.text.0
Processing 12/152: breakfast-pos.question
Processing 13/152: voicemail-pos.text.3
Processing 14/152: voicemail-zero.text.3
Processing 15/152: delayed-neg.question
Processing 16/152: goldfish-neg.text.0
Processing 17/152: delayed-neg.text.3
Processing 18/152: prize-pos.text.4
Processing 19/152: breakfast-neg.text.0
Processing 20/152: delayed-neg.interest
Processing 21/152: blackout-zero.text.1
Processing 22/152: blackout-zero.difficulty
Processing 23/152: prize-zero.text.2
Processing 24/152: goldfish-pos.interest
Processin

In [24]:
# Consolidate all features
features = pl.concat(feature_tables, how="vertical_relaxed")

# Cast to appropriate types
features = features.with_columns([
    pl.col("stimulus").cast(pl.Utf8),
    pl.col("index").cast(pl.Int64),  # Match TRT data type
    pl.col("content").cast(pl.Utf8),
    pl.col("word_len").cast(pl.Int16),
    pl.col("freq_zipf").cast(pl.Float64),
    pl.col("dep_dist").cast(pl.Int16),
    pl.col("depth").cast(pl.Int16),
    pl.col("integration_cost").cast(pl.Float64),
    pl.col("dep_label").cast(pl.Utf8),
    pl.col("pos_tag").cast(pl.Utf8),
    pl.col("lemma").cast(pl.Utf8),
    pl.col("sentence_id").cast(pl.Int32),
    pl.col("token_id_sent").cast(pl.Int32),
])

# Save features
FEATURES_PARQUET = PROC / "features_by_word.parquet"
FEATURES_CSV = PROC / "features_by_word.csv"

features.write_parquet(FEATURES_PARQUET)
features.write_csv(FEATURES_CSV)

print(f"Saved features: {features.height} rows -> {FEATURES_PARQUET}")
print(f"Columns: {features.columns}")

Saved features: 9905 rows -> data-clean\processed\features_by_word.parquet
Columns: ['stimulus', 'index', 'content', 'word_len', 'freq_zipf', 'dep_dist', 'depth', 'integration_cost', 'dep_label', 'pos_tag', 'lemma', 'sentence_id', 'token_id_sent']


In [25]:
# Merge with TRT data for modeling
print("Merging features with TRT data...")

trt_with_features = trt.join(
    features, 
    on=["stimulus", "index", "content"], 
    how="left"
)

# Save merged data
TRT_FEATURES_PARQUET = PROC / "trt_with_features.parquet"
TRT_FEATURES_CSV = PROC / "trt_with_features.csv"

trt_with_features.write_parquet(TRT_FEATURES_PARQUET)
trt_with_features.write_csv(TRT_FEATURES_CSV)

print(f"Saved merged data: {trt_with_features.height} rows -> {TRT_FEATURES_PARQUET}")

# Basic statistics
null_counts = trt_with_features.null_count()
print("\nNull counts in merged data:")
for col in ["freq_zipf", "dep_dist", "depth", "integration_cost"]:
    nulls = null_counts.select(pl.col(col)).item()
    total = trt_with_features.height
    print(f"{col}: {nulls}/{total} ({nulls/total*100:.1f}%)")

print(f"\nFinal columns: {trt_with_features.columns}")

Merging features with TRT data...
Saved merged data: 36160 rows -> data-clean\processed\trt_with_features.parquet

Null counts in merged data:
freq_zipf: 939/36160 (2.6%)
dep_dist: 102/36160 (0.3%)
depth: 102/36160 (0.3%)
integration_cost: 102/36160 (0.3%)

Final columns: ['subject_id', 'stimulus', 'condition', 'index', 'content', 'total_reading_time', 'word_len', 'freq_zipf', 'dep_dist', 'depth', 'integration_cost', 'dep_label', 'pos_tag', 'lemma', 'sentence_id', 'token_id_sent']


In [26]:
# Coverage and validation report
print("\n=== ALIGNMENT COVERAGE REPORT ===")
good_coverage = sum(1 for _, cov, _, _ in coverage_stats if cov >= 0.9)
print(f"Stimuli with ≥90% token alignment: {good_coverage}/{len(coverage_stats)}")

print("\nPer-stimulus coverage:")
for stimulus, coverage, total_tokens, aligned_tokens in sorted(coverage_stats, key=lambda x: x[1]):
    print(f"{stimulus}: {coverage:.1%} ({aligned_tokens}/{total_tokens})")

print("\n=== FEATURE SUMMARY ===")
feature_summary = features.select([
    pl.col("word_len").mean().alias("avg_word_len"),
    pl.col("freq_zipf").mean().alias("avg_freq_zipf"),
    pl.col("dep_dist").mean().alias("avg_dep_dist"),
    pl.col("depth").mean().alias("avg_depth"),
    pl.col("integration_cost").mean().alias("avg_integration_cost"),
]).to_pandas().iloc[0]

for col, val in feature_summary.items():
    print(f"{col}: {val:.2f}")

print("\n=== DEPENDENCY RELATIONS ===")
dep_counts = features.group_by("dep_label").len().sort("len", descending=True).head(10)
print(dep_counts.to_pandas())

print("\nPipeline completed successfully!")
print(f"Ready for mixed-effects modeling with {trt_with_features.height} observations")
print(f"Subjects: {trt_with_features['subject_id'].n_unique()}")
print(f"Conditions: {sorted(trt_with_features['condition'].unique().to_list())}")


=== ALIGNMENT COVERAGE REPORT ===
Stimuli with ≥90% token alignment: 152/152

Per-stimulus coverage:
prize-pos.text.0: 95.5% (84/88)
blackout-pos.text.2: 96.3% (79/82)
prize-pos.text.4: 97.0% (32/33)
prize-zero.text.0: 97.3% (108/111)
delayed-pos.text.3: 97.5% (39/40)
breakfast-pos.text.0: 97.7% (85/87)
blackout-pos.text.1: 97.7% (85/87)
breakfast-pos.text.1: 98.3% (113/115)
goldfish-pos.text.0: 98.5% (64/65)
goldfish-pos.text.2: 98.6% (68/69)
prize-pos.text.2: 98.7% (74/75)
prize-pos.text.3: 98.8% (81/82)
breakfast-pos.text.3: 98.8% (84/85)
voicemail-pos.text.1: 98.9% (88/89)
delayed-pos.text.2: 98.9% (88/89)
blackout-pos.text.0: 98.9% (91/92)
delayed-neg.text.0: 99.1% (107/108)
delayed-zero.text.0: 99.1% (107/108)
blackout-zero.text.2: 99.2% (122/123)
breakfast-zero.question: 100.0% (25/25)
prize-zero.text.4: 100.0% (74/74)
breakfast-neg.question: 100.0% (35/35)
delayed-neg.text.1: 100.0% (105/105)
blackout-zero.text.3: 100.0% (123/123)
delayed-pos.interest: 100.0% (23/23)
voicemail

# Mixed-effects pipeline (primary study model)
This is the mixed-effects pipeline.
- Outcome: log total reading time (TRT) with light trimming.
- Predictors: standardized frequency, length, and locality metrics; sum-coded condition.
- Random effects: crossed random intercepts (subject + item); optional subject slopes for frequency and length if stable/improving fit.
- Exports: fixed effects table, model choice/metrics, residual diagnostics, and simple fit summary.

In [27]:
# Imports and data load
from pathlib import Path
import json
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
import matplotlib.pyplot as plt

BASE = Path('data-clean') / 'processed'
PARQ = BASE / 'trt_with_features.parquet'
CSV = BASE / 'trt_with_features.csv'
OUT = BASE
OUT.mkdir(parents=True, exist_ok=True)

df = pl.read_parquet(PARQ) if PARQ.exists() else pl.read_csv(CSV)
use = (df
       .select(['total_reading_time','subject_id','stimulus','condition',
                'word_len','freq_zipf','dep_dist','depth','integration_cost'])
       .drop_nulls(subset=['total_reading_time','word_len','freq_zipf'])
).to_pandas()

# Ensure identifiers are strings (safer for formulas)
use['subject_id'] = use['subject_id'].astype(str)
use['stimulus'] = use['stimulus'].astype(str)

# Light trimming and log-transform (literature-aligned)
use = use[(use['total_reading_time'] >= 150) & (use['total_reading_time'] <= 4000)].copy()
use['log_trt'] = np.log(use['total_reading_time'])

# Standardize continuous predictors
for c in ['word_len','freq_zipf','dep_dist','depth','integration_cost']:
    s = use[c].std()
    use[c + '_z'] = (use[c] - use[c].mean()) / (s if s and not np.isnan(s) else 1.0)

print('Rows after trimming:', len(use))
use.head(3)

Rows after trimming: 18172


,total_reading_time,subject_id,stimulus,condition,word_len,freq_zipf,dep_dist,depth,integration_cost,log_trt,word_len_z,freq_zipf_z,dep_dist_z,depth_z,integration_cost_z
0,187,P01,blackout-zero.text.2,zero,3,6.487455,2,2,1.0,5.231109,-0.911924,0.796754,-0.129824,-0.110513,-0.123215
1,255,P01,blackout-zero.text.2,zero,5,4.501196,1,2,0.6,5.541264,-0.085272,-0.653453,-0.470300,-0.110513,-0.394412
4,255,P01,blackout-zero.text.2,zero,5,5.385928,1,3,0.6,5.541264,-0.085272,-0.007493,-0.470300,0.550312,-0.394412


In [28]:
# Fit primary model: crossed random intercepts (subject + item)
vc = {'stimulus': '0 + C(stimulus)'}
formula = 'log_trt ~ freq_zipf_z + word_len_z + dep_dist_z + depth_z + integration_cost_z + C(condition, Sum)'
m_crossed = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1', data=use)
res_crossed = m_crossed.fit(method='lbfgs', reml=True)
print('=== MixedLM (crossed intercepts) ===')
print(res_crossed.summary())

# Attempt subject slopes (freq & length) + item VC; fall back on failure
res_slopes = None
try:
    m_slopes = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1 + freq_zipf_z + word_len_z', data=use)
    res_slopes = m_slopes.fit(method='lbfgs', reml=True)
    print('=== MixedLM (subject slopes: freq, length) + item VC ===')
    print(res_slopes.summary())
except Exception as e:
    print('[Info] Subject-slopes model failed:', repr(e))

# Choose model (use ML AIC if available; otherwise slope-variance heuristic)
chosen = 'crossed'
chosen_res = res_crossed

# Try ML AIC selection without altering REML reporting
aic_ml = {'crossed': None, 'slopes': None}
try:
    m_crossed_ml = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1', data=use)
    res_crossed_ml = m_crossed_ml.fit(method='lbfgs', reml=False)
    aic_ml['crossed'] = float(getattr(res_crossed_ml, 'aic', np.nan))
    if res_slopes is not None:
        m_slopes_ml = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1 + freq_zipf_z + word_len_z', data=use)
        res_slopes_ml = m_slopes_ml.fit(method='lbfgs', reml=False)
        aic_ml['slopes'] = float(getattr(res_slopes_ml, 'aic', np.nan))
except Exception as e:
    print('[Info] ML AIC selection skipped:', repr(e))

prefer = False
if res_slopes is not None:
    # Prefer slopes if ML AIC improves
    if aic_ml['crossed'] is not None and aic_ml['slopes'] is not None and np.isfinite(aic_ml['crossed']) and np.isfinite(aic_ml['slopes']):
        prefer = aic_ml['slopes'] < aic_ml['crossed']
    # Or if slope variances are clearly > 0
    try:
        diag = np.diag(res_slopes.cov_re)
        slope_variance_positive = (len(diag) >= 3) and ((float(diag[1]) > 1e-4) or (float(diag[2]) > 1e-4))
        prefer = prefer or slope_variance_positive
    except Exception:
        pass
    if prefer:
        chosen = 'slopes'
        chosen_res = res_slopes

print(f'Chosen model: {chosen}')

# Export fixed effects and decision
fe = pd.DataFrame({'coef': chosen_res.params, 'se': chosen_res.bse})
fe.to_csv(OUT / 'mixedlm_fixed_effects_chosen.csv')
choice = {
    'chosen': chosen,
    'aic': {
        'crossed': float(getattr(res_crossed, 'aic', np.nan)),
        'slopes': float(getattr(res_slopes, 'aic', np.nan)) if res_slopes is not None else None
    },
    'aic_ml': aic_ml,
    'llf': {
        'crossed': float(res_crossed.llf),
        'slopes': float(res_slopes.llf) if res_slopes is not None else None
    }
}
with open(OUT / 'mixedlm_choice.json', 'w', encoding='utf-8') as f:
    json.dump(choice, f, indent=2)
print('Saved: mixedlm_fixed_effects_chosen.csv, mixedlm_choice.json')

# Residual diagnostics: fitted vs observed, residual histogram
try:
    fitted = chosen_res.fittedvalues
    resid = use.loc[fitted.index, 'log_trt'] - fitted
    # Scatter plot
    plt.figure(figsize=(5,4))
    plt.scatter(fitted, use.loc[fitted.index, 'log_trt'], s=4, alpha=0.3)
    plt.xlabel('Fitted log(TRT)')
    plt.ylabel('Observed log(TRT)')
    plt.title('Observed vs Fitted (log scale)')
    plt.tight_layout()
    plt.savefig(OUT / 'mixedlm_obs_vs_fitted.png', dpi=150)
    plt.close()
    # Histogram of residuals
    plt.figure(figsize=(5,4))
    plt.hist(resid, bins=50, alpha=0.8)
    plt.xlabel('Residual (log TRTs)')
    plt.ylabel('Count')
    plt.title('Residuals histogram')
    plt.tight_layout()
    plt.savefig(OUT / 'mixedlm_residuals_hist.png', dpi=150)
    plt.close()
    print('Saved: mixedlm_obs_vs_fitted.png, mixedlm_residuals_hist.png')
except Exception as e:
    print('[Info] Diagnostics plotting skipped:', repr(e))

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


=== MixedLM (crossed intercepts) ===
               Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    log_trt    
No. Observations:      18172      Method:                REML       
No. Groups:            12         Scale:                 0.1885     
Min. group size:       902        Log-Likelihood:        -11022.9610
Max. group size:       2001       Converged:             Yes        
Mean group size:       1514.3                                       
--------------------------------------------------------------------
                         Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 5.724    0.027 209.251 0.000  5.670  5.777
C(condition, Sum)[S.neg] -0.011    0.011  -0.979 0.327 -0.032  0.011
C(condition, Sum)[S.pos]  0.024    0.011   2.182 0.029  0.002  0.045
freq_zipf_z              -0.048    0.005  -9.699 0.000 -0.057 -0.038
word_len_z   

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


=== MixedLM (subject slopes: freq, length) + item VC ===
                 Mixed Linear Model Regression Results
Model:                 MixedLM      Dependent Variable:      log_trt    
No. Observations:      18172        Method:                  REML       
No. Groups:            12           Scale:                   0.1860     
Min. group size:       902          Log-Likelihood:          -10921.4339
Max. group size:       2001         Converged:               Yes        
Mean group size:       1514.3                                           
------------------------------------------------------------------------
                             Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercept                     5.729    0.029 200.058 0.000  5.672  5.785
C(condition, Sum)[S.neg]     -0.014    0.011  -1.257 0.209 -0.035  0.008
C(condition, Sum)[S.pos]      0.028    0.011   2.583 0.010  0.007  0.050
freq_zipf_z 

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Chosen model: slopes
Saved: mixedlm_fixed_effects_chosen.csv, mixedlm_choice.json
Saved: mixedlm_obs_vs_fitted.png, mixedlm_residuals_hist.png


# Condition effects analysis (adjusted means and contrasts)

In [29]:
# Load data, mirror preprocessing, and compute condition means/contrasts
from pathlib import Path
import json, numpy as np, pandas as pd
import polars as pl
import statsmodels.api as sm
from patsy import dmatrix

BASE = Path("data-clean") / "processed"
PARQ = BASE / "trt_with_features.parquet"
CSV = BASE / "trt_with_features.csv"
OUT = BASE
OUT.mkdir(parents=True, exist_ok=True)

# Load and mirror pipeline preprocessing
df = pl.read_parquet(PARQ) if PARQ.exists() else pl.read_csv(CSV)
use = (df
       .select(["total_reading_time","subject_id","stimulus","condition",
                "word_len","freq_zipf","dep_dist","depth","integration_cost"])
       .drop_nulls(subset=["total_reading_time","word_len","freq_zipf"])
      ).to_pandas()

use["subject_id"] = use["subject_id"].astype(str)
use["stimulus"]   = use["stimulus"].astype(str)
use = use[(use["total_reading_time"] >= 150) & (use["total_reading_time"] <= 4000)].copy()
use["log_trt"] = np.log(use["total_reading_time"])
for c in ["word_len","freq_zipf","dep_dist","depth","integration_cost"]:
    s = use[c].std()
    use[c + "_z"] = (use[c] - use[c].mean()) / (s if s and not np.isnan(s) else 1.0)

# Read chosen model and refit to get FE covariance
choice_path = OUT / "mixedlm_choice.json"
chosen = "crossed"
try:
    with choice_path.open() as f:
        chosen = json.load(f).get("chosen", "crossed")
except FileNotFoundError:
    print("[Info] mixedlm_choice.json not found; defaulting to crossed intercepts")

formula_fe = "log_trt ~ freq_zipf_z + word_len_z + dep_dist_z + depth_z + integration_cost_z + C(condition, Sum)"
vc = {"stimulus": "0 + C(stimulus)"}
if chosen == "slopes":
    m = sm.MixedLM.from_formula(formula_fe, groups="subject_id", vc_formula=vc,
                                re_formula="1 + freq_zipf_z + word_len_z", data=use)
else:
    m = sm.MixedLM.from_formula(formula_fe, groups="subject_id", vc_formula=vc,
                                re_formula="1", data=use)
res = m.fit(method="lbfgs", reml=True)
fe = res.fe_params
cov_fe = res.cov_params()
# Keep only the fixed-effects block to align with X columns
if isinstance(cov_fe, np.ndarray):
    cov_fe = pd.DataFrame(cov_fe, index=fe.index, columns=fe.index)
else:
    cov_fe = cov_fe.loc[fe.index, fe.index]

# Build design rows per condition at z=0
levels = pd.Index(sorted(use["condition"].unique()))
tmpl = pd.DataFrame({
    "freq_zipf_z":[0.0]*len(levels),
    "word_len_z":[0.0]*len(levels),
    "dep_dist_z":[0.0]*len(levels),
    "depth_z":[0.0]*len(levels),
    "integration_cost_z":[0.0]*len(levels),
    "condition": levels
})
X = dmatrix("1 + freq_zipf_z + word_len_z + dep_dist_z + depth_z + integration_cost_z + C(condition, Sum)",
            tmpl, return_type="dataframe")
X = X.reindex(columns=fe.index, fill_value=0.0)

# Marginal means per condition (fixed effects)
mu = (X.values @ fe.values)
se = np.sqrt(np.einsum("ij,jk,ik->i", X.values, cov_fe.values, X.values))
lo, hi = mu - 1.96*se, mu + 1.96*se
means = pd.DataFrame({
    "condition": levels,
    "log_mean": mu,
    "log_se": se,
    "log_ci_lo": lo,
    "log_ci_hi": hi,
})
means["pct_mean"]  = (np.exp(means["log_mean"]) - 1.0) * 100.0
means["pct_ci_lo"] = (np.exp(means["log_ci_lo"]) - 1.0) * 100.0
means["pct_ci_hi"] = (np.exp(means["log_ci_hi"]) - 1.0) * 100.0

# Pairwise contrasts
rows = []
for i in range(len(levels)):
    for j in range(i+1, len(levels)):
        ci, cj = levels[i], levels[j]
        cvec = (X.iloc[i] - X.iloc[j]).values
        est = float(cvec @ fe.values)
        se  = float(np.sqrt(cvec @ cov_fe.values @ cvec))
        lo, hi = est - 1.96*se, est + 1.96*se
        rows.append({
            "contrast": f"{ci} - {cj}",
            "log_diff": est,
            "log_se": se,
            "log_ci_lo": lo,
            "log_ci_hi": hi,
            "pct_diff": (np.exp(est) - 1.0) * 100.0,
            "pct_ci_lo": (np.exp(lo) - 1.0) * 100.0,
            "pct_ci_hi": (np.exp(hi) - 1.0) * 100.0,
        })
contrasts = pd.DataFrame(rows)

# Save
means.to_csv(OUT / "condition_marginal_means.csv", index=False)
contrasts.to_csv(OUT / "condition_pairwise_contrasts.csv", index=False)
print("Saved:", OUT / "condition_marginal_means.csv", "|", OUT / "condition_pairwise_contrasts.csv")

# Small summary
print("\nMarginal means (%):\n", means[["condition","pct_mean","pct_ci_lo","pct_ci_hi"]].round(2))
print("\nPairwise contrasts (%):\n", contrasts[["contrast","pct_diff","pct_ci_lo","pct_ci_hi"]].round(2))

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Saved: data-clean\processed\condition_marginal_means.csv | data-clean\processed\condition_pairwise_contrasts.csv

Marginal means (%):
   condition  pct_mean  pct_ci_lo  pct_ci_hi
0       neg  30236.92   28475.85   32106.53
1       pos  31533.91   29679.86   33503.39
2      zero  30204.45   28449.23   32067.59

Pairwise contrasts (%):
      contrast  pct_diff  pct_ci_lo  pct_ci_hi
0   neg - pos     -4.10      -7.62      -0.45
1  neg - zero      0.11      -3.42       3.77
2  pos - zero      4.39       0.66       8.25


# Linguistic feature differences across gaze conditions

Goals:
- Describe how features (frequency, length, dependency distance, depth, integration cost) differ by condition (neg/zero/pos).
- Test if condition predicts each feature (mixed-effects).
- Optionally compare TRT condition effects with and without features.

In [30]:
from pathlib import Path
import warnings
import json
import math

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Paths
BASE = Path('data-clean') / 'processed'
PQ = BASE / 'trt_with_features.parquet'
CSV = BASE / 'trt_with_features.csv'
OUT = BASE
OUT.mkdir(parents=True, exist_ok=True)

# Load
df = pl.read_parquet(PQ) if PQ.exists() else pl.read_csv(CSV)
use = (
    df.select([
        'total_reading_time','subject_id','stimulus','condition',
        'word_len','freq_zipf','dep_dist','depth','integration_cost'
    ])
    .drop_nulls(subset=['total_reading_time','word_len','freq_zipf'])
).to_pandas()

# Trim TRT and transform
use = use[(use['total_reading_time'] >= 150) & (use['total_reading_time'] <= 4000)].copy()
use['log_trt'] = np.log(use['total_reading_time'])

# Standardize features globally
FEATURES = ['freq_zipf','word_len','dep_dist','depth','integration_cost']
for c in FEATURES:
    mu = use[c].mean(); sd = use[c].std()
    use[c + '_z'] = (use[c] - mu) / (sd if sd and not np.isnan(sd) else 1.0)

# Basic integrity checks
conds = sorted(use['condition'].unique().tolist())
print('Conditions:', conds)
print('Rows:', len(use), '| Subjects:', use['subject_id'].nunique(), '| Stimuli:', use['stimulus'].nunique())
assert set(conds) == set(['neg','pos','zero']), 'Expected three conditions: neg,pos,zero'
print('Setup complete.')

Conditions: ['neg', 'pos', 'zero']
Rows: 18172 | Subjects: 12 | Stimuli: 152
Setup complete.


In [31]:
# Descriptive summaries by condition
summ_rows = []
for c in FEATURES:
    g = use.groupby('condition')[c].agg(['count','mean','std','median'])
    g['feature'] = c
    g = g.reset_index()[['feature','condition','count','mean','std','median']]
    summ_rows.append(g)
summ = pd.concat(summ_rows, ignore_index=True)
summ.to_csv(OUT / 'feature_condition_descriptives.csv', index=False)
print('Saved descriptive stats ->', OUT / 'feature_condition_descriptives.csv')

# Plots: boxplots for each feature across conditions
for c in FEATURES:
    fig, ax = plt.subplots(figsize=(6,4))
    data = [use.loc[use['condition']==k, c].dropna().values for k in ['neg','zero','pos']]
    ax.boxplot(data, labels=['neg','zero','pos'], showfliers=False)
    ax.set_title(f'{c} by condition')
    ax.set_ylabel(c)
    fig.tight_layout()
    p = OUT / f'feature_boxplot_{c}.png'
    fig.savefig(p, dpi=150)
    plt.close(fig)
print('Saved boxplots to', OUT)

Saved descriptive stats -> data-clean\processed\feature_condition_descriptives.csv


C:\Users\Mert Yeşilyurt\AppData\Local\Temp\ipykernel_9560\2890229590.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=['neg','zero','pos'], showfliers=False)
C:\Users\Mert Yeşilyurt\AppData\Local\Temp\ipykernel_9560\2890229590.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=['neg','zero','pos'], showfliers=False)
C:\Users\Mert Yeşilyurt\AppData\Local\Temp\ipykernel_9560\2890229590.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=['neg','zero','pos'], showfliers=False)
C:\Users\Mert Yeşilyurt\AppData\Local\Temp\ipykernel_9560\289022959

Saved boxplots to data-clean\processed


C:\Users\Mert Yeşilyurt\AppData\Local\Temp\ipykernel_9560\2890229590.py:16: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(data, labels=['neg','zero','pos'], showfliers=False)


In [32]:
# Mixed-effects: feature_z ~ C(condition, Sum) with subject intercept and stimulus VC
vc = {'stimulus': '0 + C(stimulus)'}
rows = []
for c in FEATURES:
    y = c + '_z'
    formula = f"{y} ~ C(condition, Sum)"
    try:
        m = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1', data=use)
        res = m.fit(method='lbfgs', reml=True)
        # Extract condition coefficients and CIs
        coefs = res.params
        ses = res.bse
        ci = res.conf_int()
        # Save per-feature details
        ci_lo_col = 0 if 0 in ci.columns else ci.columns[0]
        ci_hi_col = 1 if 1 in ci.columns else ci.columns[1]
        df_out = pd.DataFrame({
            'term': coefs.index,
            'coef': coefs.values,
            'se': ses.values,
            'ci_lo': ci[ci_lo_col].values,
            'ci_hi': ci[ci_hi_col].values
        })
        df_out.to_csv(OUT / f'feature_mixedlm_{c}.csv', index=False)
        # Add to combined summary only condition terms
        keep = [t for t in coefs.index if t.startswith('C(condition, Sum)')]
        for t in keep:
            rows.append({
                'feature': c,
                'term': t,
                'coef': float(coefs[t]),
                'se': float(ses[t]),
                'ci_lo': float(ci.loc[t, ci_lo_col]),
                'ci_hi': float(ci.loc[t, ci_hi_col])
            })
        print(f'[OK] {c} model fit; saved ->', OUT / f'feature_mixedlm_{c}.csv')
    except Exception as e:
        print(f'[Warn] {c} model failed:', repr(e))

combined = pd.DataFrame(rows)
combined.to_csv(OUT / 'feature_condition_mixedlm_summary.csv', index=False)
print('Saved combined feature~condition summary ->', OUT / 'feature_condition_mixedlm_summary.csv')

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


[OK] freq_zipf model fit; saved -> data-clean\processed\feature_mixedlm_freq_zipf.csv


C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


[OK] word_len model fit; saved -> data-clean\processed\feature_mixedlm_word_len.csv


C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


[OK] dep_dist model fit; saved -> data-clean\processed\feature_mixedlm_dep_dist.csv


C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


[OK] depth model fit; saved -> data-clean\processed\feature_mixedlm_depth.csv


C:\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 108.148520
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


[OK] integration_cost model fit; saved -> data-clean\processed\feature_mixedlm_integration_cost.csv
Saved combined feature~condition summary -> data-clean\processed\feature_condition_mixedlm_summary.csv


In [33]:
# Optional: TRT condition effects with and without features
def pct(b):
    return 100.0 * (math.exp(b) - 1.0)

try:
    # Without features
    m0 = sm.MixedLM.from_formula('log_trt ~ C(condition, Sum)', groups='subject_id', vc_formula={'stimulus': '0 + C(stimulus)'}, re_formula='1', data=use).fit(method='lbfgs', reml=True)
    coefs0 = m0.params[m0.params.index.str.startswith('C(condition, Sum)')]
    out0 = pd.DataFrame({'term': coefs0.index, 'coef': coefs0.values})
    out0['pct'] = out0['coef'].apply(pct)
    out0.to_csv(OUT / 'trt_condition_effects_without_features.csv', index=False)

    # With features
    m1 = sm.MixedLM.from_formula('log_trt ~ C(condition, Sum) + freq_zipf_z + word_len_z + dep_dist_z + depth_z + integration_cost_z', groups='subject_id', vc_formula={'stimulus': '0 + C(stimulus)'}, re_formula='1 + freq_zipf_z + word_len_z', data=use).fit(method='lbfgs', reml=True)
    coefs1 = m1.params[m1.params.index.str.startswith('C(condition, Sum)')]
    out1 = pd.DataFrame({'term': coefs1.index, 'coef': coefs1.values})
    out1['pct'] = out1['coef'].apply(pct)
    out1.to_csv(OUT / 'trt_condition_effects_with_features.csv', index=False)
    print('Saved TRT condition effect tables (with/without features).')
except Exception as e:
    print('[Info] TRT with/without features block skipped due to error:', repr(e))

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Saved TRT condition effect tables (with/without features).


Done. Outputs are saved under `data-clean/processed/`:
- `feature_condition_descriptives.csv`
- `feature_boxplot_*.png`
- `feature_mixedlm_*.csv` and combined `feature_condition_mixedlm_summary.csv`
- `trt_condition_effects_without_features.csv` and `trt_condition_effects_with_features.csv` (if successful)

# Reading study — compact report

This report summarizes:
1) Main: Effects of lexical/syntactic features on log(TRT).
2) Secondary 1: Feature differences across conditions.
3) Secondary 2: TRT differences across conditions.

Figures: (A) Forest plot of feature effects (percent change) and (B) Condition marginal means with 95% CIs.

In [34]:
from pathlib import Path
import math
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('data-clean') / 'processed'
OUT = BASE  # save alongside existing artifacts
OUT.mkdir(parents=True, exist_ok=True)

paths = {
  'fe_validation': BASE / 'mixedlm_fe_summary_validation.csv',
  'fe_chosen': BASE / 'mixedlm_fixed_effects_chosen.csv',
  'cond_means': BASE / 'condition_marginal_means.csv',
  'cond_contrasts': BASE / 'condition_pairwise_contrasts.csv',
  'feat_desc': BASE / 'feature_condition_descriptives.csv',
  'feat_mixed': BASE / 'feature_condition_mixedlm_summary.csv'
}
for k, p in paths.items():
    print(f'{k}:', p.exists(), '->', p)
print('Setup complete.')

fe_validation: True -> data-clean\processed\mixedlm_fe_summary_validation.csv
fe_chosen: True -> data-clean\processed\mixedlm_fixed_effects_chosen.csv
cond_means: True -> data-clean\processed\condition_marginal_means.csv
cond_contrasts: True -> data-clean\processed\condition_pairwise_contrasts.csv
feat_desc: True -> data-clean\processed\feature_condition_descriptives.csv
feat_mixed: True -> data-clean\processed\feature_condition_mixedlm_summary.csv
Setup complete.


In [35]:
# 1) Main: Feature effects on log(TRT) — table (percent change with 95% CI)
def pct(x):
    return 100.0 * (math.exp(x) - 1.0)

terms = ['freq_zipf_z','word_len_z','dep_dist_z','depth_z','integration_cost_z']
labels = {
    'freq_zipf_z':'Frequency (Zipf)',
    'word_len_z':'Word length',
    'dep_dist_z':'Dependency distance',
    'depth_z':'Syntactic depth',
    'integration_cost_z':'Integration cost'
}

# Prefer validation summary (has pct_change columns)
if paths['fe_validation'].exists():
    fe = pd.read_csv(paths['fe_validation'])
    fe = fe[fe['term'].isin(terms)].copy()
    # Expect columns: term, estimate, lo95, hi95, pct_change_est, pct_change_lo, pct_change_hi
    required = {'term','estimate','lo95','hi95','pct_change_est','pct_change_lo','pct_change_hi'}
    if not required.issubset(fe.columns):
        raise ValueError('mixedlm_fe_summary_validation.csv missing expected columns')
    fe['label'] = fe['term'].map(labels)
    tbl_main = fe[['label','pct_change_est','pct_change_lo','pct_change_hi','estimate','lo95','hi95']]\
        .rename(columns={
            'pct_change_est':'percent',
            'pct_change_lo':'percent_lo',
            'pct_change_hi':'percent_hi',
            'estimate':'beta',
            'lo95':'ci_lo',
            'hi95':'ci_hi'
        }).sort_values('label').reset_index(drop=True)
elif paths['fe_chosen'].exists():
    # Fallback: compute percent from beta and CI if present
    fe = pd.read_csv(paths['fe_chosen'])
    fe = fe[fe['term'].isin(terms)].copy()
    fe['label'] = fe['term'].map(labels)
    if {'coef','ci_lo','ci_hi'}.issubset(fe.columns):
        fe['percent'] = fe['coef'].apply(pct)
        fe['percent_lo'] = fe['ci_lo'].apply(pct)
        fe['percent_hi'] = fe['ci_hi'].apply(pct)
        tbl_main = fe[['label','percent','percent_lo','percent_hi','coef','ci_lo','ci_hi']]\
            .rename(columns={'coef':'beta'})
    elif 'coef' in fe.columns:
        fe = fe.rename(columns={'coef':'beta'})
        fe['percent'] = fe['beta'].apply(pct)
        tbl_main = fe[['label','percent','beta']]
    else:
        raise ValueError('mixedlm_fixed_effects_chosen.csv missing coef columns.')
else:
    raise FileNotFoundError('No FE summary file found for main table.')

# Save and display
tbl_main.to_csv(OUT / 'report_main_feature_effects.csv', index=False)
tbl_main

,label,percent,percent_lo,percent_hi,beta,ci_lo,ci_hi
0,Dependency distance,-0.783611,-1.771351,0.214061,-0.007867,-0.017872,0.002138
1,Frequency (Zipf),-5.290908,-7.830629,-2.681206,-0.054360,-0.081542,-0.027178
2,Integration cost,0.030052,-0.962779,1.032836,0.000300,-0.009674,0.010275
3,Syntactic depth,1.329408,0.666388,1.996795,0.013206,0.006642,0.019771
4,Word length,9.356613,6.894830,11.875092,0.089444,0.066675,0.112213


In [36]:
# 2) Secondary 1: Feature differences across conditions — tables
# (a) MixedLM condition effects on standardized features
p = paths['feat_mixed']
assert p.exists(), f'Missing {p}'
feat_mixed = pd.read_csv(p)
# Parse condition labels from terms like C(condition, Sum)[S.pos]
def term_to_condition(t):
    if 'S.pos' in t:
        return 'pos'
    if 'S.neg' in t:
        return 'neg'
    return 'zero?'
feat_mixed['condition'] = feat_mixed['term'].apply(term_to_condition)
feat_mixed = feat_mixed[['feature','condition','coef','ci_lo','ci_hi']].copy()
feat_mixed = feat_mixed.sort_values(['feature','condition']).reset_index(drop=True)
feat_mixed.to_csv(OUT / 'report_feature_condition_effects.csv', index=False)
feat_mixed.head(10)

,feature,condition,coef,ci_lo,ci_hi
0,dep_dist,neg,0.025457,0.004831,0.046083
1,dep_dist,pos,-0.014373,-0.035691,0.006946
2,depth,neg,0.143050,0.111763,0.174338
3,depth,pos,-0.129509,-0.161418,-0.097599
4,freq_zipf,neg,0.324625,0.299413,0.349837
5,freq_zipf,pos,-0.431886,-0.458376,-0.405396
6,integration_cost,neg,0.049957,0.028674,0.071240
7,integration_cost,pos,-0.046412,-0.068561,-0.024263
8,word_len,neg,-0.305453,-0.332644,-0.278262
9,word_len,pos,0.398906,0.370695,0.427117


In [37]:
# (b) Descriptive means by condition for each feature
p = paths['feat_desc']
assert p.exists(), f'Missing {p}'
feat_desc = pd.read_csv(p)
wide_means = (feat_desc.pivot_table(index='feature', columns='condition', values='mean', aggfunc='first'))
if all(c in wide_means.columns for c in ['neg','zero','pos']):
    wide_means = wide_means[['neg','zero','pos']]
wide_means.to_csv(OUT / 'report_feature_condition_descriptives_wide.csv')
wide_means

condition,neg,zero,pos
feature,,,
dep_dist,2.449349,2.347391,2.339787
depth,2.401642,2.138148,1.926000
freq_zipf,5.828690,5.524684,4.747110
integration_cost,1.252029,1.174212,1.108051
word_len,4.468247,5.013411,6.284422


In [38]:
# 3) Secondary 2: TRT differences across conditions — tables
p_means = paths['cond_means']
p_contr = paths['cond_contrasts']
assert p_means.exists(), f'Missing {p_means}'
assert p_contr.exists(), f'Missing {p_contr}'
cond_means = pd.read_csv(p_means)[['condition','pct_mean','pct_ci_lo','pct_ci_hi']].copy().sort_values('condition')
cond_means.to_csv(OUT / 'report_condition_marginal_means.csv', index=False)

cond_contrasts = pd.read_csv(p_contr)[['contrast','pct_diff','pct_ci_lo','pct_ci_hi']].copy()
cond_contrasts.to_csv(OUT / 'report_condition_pairwise_contrasts.csv', index=False)
cond_means, cond_contrasts

(  condition      pct_mean     pct_ci_lo     pct_ci_hi
 0       neg  30236.921236  28475.847295  32106.526742
 1       pos  31533.910959  29679.860224  33503.392192
 2      zero  30204.454268  28449.232400  32067.588103,
      contrast  pct_diff  pct_ci_lo  pct_ci_hi
 0   neg - pos -4.099998  -7.615254  -0.450986
 1  neg - zero  0.107136  -3.423179   3.766500
 2  pos - zero  4.387001   0.661835   8.250023)

In [ ]:
# Ensure seaborn is available right before plotting
import seaborn as sns  # idempotent import


In [ ]:
# Figure A: Forest plot for main feature effects (percent with 95% CI)
fig, ax = plt.subplots(figsize=(6, 3.8))
d = tbl_main.copy()
d = d.sort_values('label')
y = np.arange(len(d))
ax.errorbar(d['percent'], y, xerr=[d['percent']-d.get('percent_lo', d['percent']), d.get('percent_hi', d['percent'])-d['percent']], fmt='o', color='black', ecolor='gray', capsize=3)
ax.axvline(0, color='red', lw=1, alpha=0.6)
ax.set_yticks(y); ax.set_yticklabels(d['label'])
ax.set_xlabel('Percent change in TRT per +1 SD (95% CI)')
ax.set_title('Feature effects on reading time')
fig.tight_layout()
fig.savefig(OUT / 'report_fig_feature_effects.png', dpi=150)
plt.close(fig)
print('Saved figure ->', OUT / 'report_fig_feature_effects.png')

Saved figure -> data-clean\processed\report_fig_feature_effects.png


In [ ]:
# Figure B: Condition marginal means with 95% CI (percent scale)
cm = cond_means.copy()
order = ['neg','zero','pos']
cm['condition'] = pd.Categorical(cm['condition'], categories=order, ordered=True)
cm = cm.sort_values('condition')
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.bar(cm['condition'], cm['pct_mean'], color=['#4C78A8','#72B7B2','#F58518'])
# error bars
yerr = np.vstack([cm['pct_mean'] - cm['pct_ci_lo'], cm['pct_ci_hi'] - cm['pct_mean']])
ax.errorbar(cm['condition'], cm['pct_mean'], yerr=yerr, fmt='none', ecolor='black', capsize=4)
ax.set_ylabel('Adjusted TRT (percent scale)')
ax.set_title('Condition marginal means (95% CI)')
fig.tight_layout()
fig.savefig(OUT / 'report_fig_condition_means.png', dpi=150)
plt.close(fig)  
print('Saved figure ->', OUT / 'report_fig_condition_means.png')

Saved figure -> data-clean\processed\report_fig_condition_means.png


Outputs written under `data-clean/processed/`:
- Tables: `report_main_feature_effects.csv`, `report_feature_condition_effects.csv`, `report_feature_condition_descriptives_wide.csv`,
  `report_condition_marginal_means.csv`, `report_condition_pairwise_contrasts.csv`
- Figures: `report_fig_feature_effects.png`, `report_fig_condition_means.png`